# 20.15 兩代Omni怎樣分工理解、文字回答與聲音生成？


問「聲音響起時，畫面是哪個形狀？」，回答「圓形」，再把答案變成聲音，需要三件不同的工作：從輸入取得線索、決定回答內容、產生可以播放的波形。第19章的設計把這條路拆成自己的小模組，驗收時要查各站和整條鏈。現在對照Qwen2.5-Omni與Qwen3-Omni，追蹤它們把哪些資訊交給哪個零件；不以名稱相同就推論兩代接口相同。

兩代都先用影像與聲音編碼器，把像素、waveform換成表示；文字則經tokenizer。Thinker是讀取這些表示並生成文字的核心，Talker負責產生聲音代碼，再由解碼器還原波形。因此「一套模型」仍可包含多個子網路。Thinker這個角色名稱本身，也不表示每次回答都會輸出一段推理過程。

下圖實線箭頭表示資料傳遞，兩列用相同次序排：輸入、Thinker、送給Talker的條件、聲音代碼、波形。需要比較的是中間條件和最後解碼路線，不是框的大小。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-20-architecture.svg")))

2.5 Talker同時接收Thinker的高階文字表示，以及已抽樣文字token的embedding。前者保留上下文中的語意與語氣資訊；但相近語意的詞可能讀音不同，所以仍需要後者指明實際選了哪些字。3則解耦這份高階文字表示，仍以歷史文字、這輪逐步送出的文字與音影條件合成聲音。這讓外部程式可以先處理文字，再交給Talker，也能分開設定回答文字與聲音風格；不能把「不再接高階文字表示」讀成「完全不需要文字」。

兩代處理影音時間的設定也不同。2.5報告以40ms作時間座標單位，把影音表示每2秒分塊交錯；3的AuT聲音編碼器約每80ms輸出一份表示，影音直接按實際時間ID對齊。3還調整了時間、高度、寬度的旋轉角分配，接到[14.19](https://birdhackor.github.io/tiny-perceptron-vlm/14.19.html)的三軸位置。這些數字各屬其報告設定；本課的局部窗口不同，不能把小模型成績當作官方配置的重現。

聲音輸出路線的改變，則接到[17.17](https://birdhackor.github.io/tiny-perceptron-vlm/17.17.html)的殘差碼本。2.5把聲音碼交給產生mel頻譜的Transformer網路，報告稱Flow-Matching DiT，再由BigVGAN把頻譜變波形。它的滑動視窗包含一個未來block，要先等到該block才有足夠上下文。3由Talker先預測目前音框的第0個碼本，再讓MTP小網路逐次補同一音框的其餘碼本，最後交給只看左側上下文的因果卷積網路Code2Wav。這裡MTP沿碼本軸補細節，與[7.20](https://birdhackor.github.io/tiny-perceptron-vlm/7.20.html)預測未來文字位置的MTP不同。

3的Thinker與Talker都使用MoE。稀疏啟用節省的是一部分計算，沒有讓未啟用的權重自動消失。下面只用「約300億需要載入、每token約啟用30億」作名稱量級的乘法例，假設每個權重2 bytes、不卸載到其他裝置；不是精確參數盤點或執行模型。


In [ ]:
total_parameters = 30_000_000_000
active_parameters = 3_000_000_000
bytes_per_parameter = 2
stored = total_parameters * bytes_per_parameter
active_only = active_parameters * bytes_per_parameter
print("全部權重名義值GB", stored / 10**9)
print("全部權重名義值GiB", round(stored / 1024**3, 2))
print("只按啟用量算GB", active_only / 10**9)

輸出是60.0 GB、55.88 GiB和6.0 GB；最後一項只對啟用量做乘法，不能用來宣稱6 GB顯存足夠。真實執行還要盤點所有子網路、KV cache與暫存空間，不能重複計算已包含在參數總數裡的部分。官方模型卡另有BF16、FlashAttention2（注意力計算的實作）與指定影片長度的理論需求表，也不是本課的實測。讀報告和自己訓練小模型，均不要求先把30B載入本機。

練習把每權重bytes改成4，先算三個輸出都會加倍，再核對。這只改了儲存精度；沒有因此改變每token選幾個expert，也不能推論速度加倍。

原始依據：[Qwen2.5-Omni報告v1](https://arxiv.org/html/2503.20215v1)§2.1–2.4；[Qwen3-Omni報告v1](https://arxiv.org/html/2509.17765v1)§2.1–2.5。2.5原報告的7B比較分數與後來3B的實際config分開；本課不把3B實測當成30B架構收益。
